# ISRO BAH 2026 — PS-07: Notebook 5: TLS/BLS Detection + SDE Gating

**Purpose**: Run Transit Least Squares on all 60k preprocessed light curves, apply iterative multi-planet search, validate with BLS, and gate by SDE tier.  
**Requirements**: DET-01, DET-02, DET-03, DET-04, DET-05  
**Input dataset**: `tess-preprocessed` (mount in Kaggle settings)  
**Estimated runtime**: 8–12 hours (4-core Kaggle CPU, checkpointed)  
**Output**: `tls_candidates.parquet` → final Phase 1 deliverable

**Checkpoint**: Progress saved every 1000 stars — safe to interrupt and re-run.  
**Final cell**: Smoke test — recovers WASP-121b, L 98-59 (3 planets), TOI-270 (3 planets).

In [ ]:
%pip install -q "transitleastsquares>=1.0.31" "astropy>=5.3" "pyarrow>=14" "lightkurve>=2.4"


In [ ]:
import sys, os
from pathlib import Path

PIPELINE_PATH = Path('/kaggle/input/isro-bah-pipeline')
if PIPELINE_PATH.exists():
    sys.path.insert(0, str(PIPELINE_PATH))
else:
    CLONE_DIR = Path('/kaggle/working/ISRO-BAH')
    if not CLONE_DIR.exists():
        GITHUB_URL = 'https://github.com/YOUR_USERNAME/ISRO-BAH.git'  # ← UPDATE THIS
        subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, str(CLONE_DIR)], check=True)
    sys.path.insert(0, str(CLONE_DIR))

import pipeline.config as cfg
from pathlib import Path
import shutil

KAGGLE_OUT = Path('/kaggle/working')
cfg.OUTPUTS_DIR   = KAGGLE_OUT / 'outputs'
cfg.RAW_DIR       = cfg.OUTPUTS_DIR / 'raw'
cfg.PREP_DIR      = cfg.OUTPUTS_DIR / 'preprocessed'
cfg.CATALOGUE_DIR = cfg.OUTPUTS_DIR / 'catalogue'
cfg.MASTER_CATALOGUE_PATH = cfg.CATALOGUE_DIR / 'master_catalogue.parquet'
cfg.TLS_RESULTS_PATH      = cfg.CATALOGUE_DIR / 'tls_candidates.parquet'

for d in [cfg.PREP_DIR, cfg.CATALOGUE_DIR]: d.mkdir(parents=True, exist_ok=True)

# Mount preprocessed input
PREP_INPUT = Path('/kaggle/input/tess-preprocessed/outputs/preprocessed')
if PREP_INPUT.exists():
    files = list(PREP_INPUT.glob('*.npz'))
    for f in files:
        link = cfg.PREP_DIR / f.name
        if not link.exists():
            link.symlink_to(f)
    print(f'Mounted {len(files)} preprocessed .npz files')
else:
    print('⚠ tess-preprocessed input dataset not found. Add it in Kaggle settings.')

# Also mount catalogue
cat_input = Path('/kaggle/input/tess-preprocessed/outputs/catalogue/master_catalogue.parquet')
if cat_input.exists():
    shutil.copy(str(cat_input), str(cfg.MASTER_CATALOGUE_PATH))
    print(f'Master catalogue copied.')

In [ ]:
# Build star list from catalogue
import pandas as pd

cat = pd.read_parquet(str(cfg.MASTER_CATALOGUE_PATH))
# Only process stars that passed filters
processable = cat[
    (cat['download_status'] == 'SUCCESS') &
    (cat['exclusion_reason'].isna() | (cat['exclusion_reason'] == 'NONE'))
].copy()

star_list = [
    (
        int(row['tic_id']),
        int(row['sector']),
        float(row.get('u1', cfg.LD_DEFAULT[0]) or cfg.LD_DEFAULT[0]),
        float(row.get('u2', cfg.LD_DEFAULT[1]) or cfg.LD_DEFAULT[1]),
    )
    for _, row in processable.iterrows()
]
print(f'Stars to process with TLS: {len(star_list)}')
print(f'  Sector 1: {sum(1 for s in star_list if s[1]==1)}')
print(f'  Sector 2: {sum(1 for s in star_list if s[1]==2)}')
print(f'  Sector 3: {sum(1 for s in star_list if s[1]==3)}')

In [ ]:
# ── MAIN: TLS batch run (checkpointed) ───────────────────────────────────
# ⚠ This runs for 8–12 hours. Checkpoint saves every 1000 stars.
# Re-running skips already-processed stars.
import time
from pipeline.detect.tls_search import run_tls_batch

# Override checkpoint path to Kaggle working dir
import pipeline.detect.tls_search as tls_mod
tls_mod.CHECKPOINT_PATH = cfg.OUTPUTS_DIR / 'tls_checkpoint.json'

print(f'Starting TLS at {time.strftime("%H:%M:%S")}')
print(f'Config: period_min={cfg.TLS_PERIOD_MIN}d, period_max={cfg.TLS_PERIOD_MAX}d')
print(f'        oversampling={cfg.TLS_OVERSAMPLING}, iterations={cfg.TLS_ITERATIONS}')

tls_results = run_tls_batch(star_list, workers=None)  # uses all CPUs

print(f'\n✓ TLS complete at {time.strftime("%H:%M:%S")}')
print(f'Total signals detected: {len(tls_results)}')

In [ ]:
# ── BLS validation on SDE≥7 candidates ────────────────────────────────────
from pipeline.detect.bls_validate import validate_candidates
from pipeline.ingest.store import load_lc_npz

full_pipeline = [r for r in tls_results if r['sde'] >= cfg.SDE_SUBTHRESHOLD]
print(f'SDE≥7 candidates for BLS validation: {len(full_pipeline)}')

# Load preprocessed LCs for BLS candidates
lc_data = {}
for r in full_pipeline:
    key = (int(r['tic_id']), int(r['sector']))
    if key not in lc_data:
        try:
            lc_data[key] = load_lc_npz(key[0], key[1], kind='preprocessed')
        except FileNotFoundError:
            pass

full_pipeline = validate_candidates(full_pipeline, lc_data)
bls_consistent = sum(1 for r in full_pipeline if r.get('bls_consistent', False))
print(f'BLS consistent with TLS: {bls_consistent}/{len(full_pipeline)}')

In [ ]:
# ── SDE gating + save ─────────────────────────────────────────────────────
from pipeline.detect.sde_gate import apply_sde_gating, save_tls_results

df_all = apply_sde_gating(tls_results)
out_path = save_tls_results(df_all)

print(f'\nTLS results saved: {len(df_all)} signals → {out_path}')
print(df_all['disposition'].value_counts())
print('\nTop 10 by SDE:')
print(df_all.head(10)[['tic_id','sector','planet_num','period','sde','snr','cdpp','bls_consistent','disposition']].to_string())

In [ ]:
# ── SMOKE TEST: Phase 1 Acceptance Criteria ───────────────────────────────
from pipeline.validate.smoke_test import run_smoke_test

results = run_smoke_test(catalogue_path=cfg.TLS_RESULTS_PATH)
all_passed = all(v['recovered'] for v in results.values())

if all_passed:
    print('\n🎉 PHASE 1 COMPLETE — ALL TARGETS RECOVERED')
else:
    missed = [name for name, v in results.items() if not v['recovered']]
    print(f'\n⚠ Phase 1 partial: missed targets: {missed}')
    print('Check TLS parameters and verify input data quality.')

In [ ]:
# ── SDE distribution plot ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SDE histogram
sdes = df_all['sde'].values
axes[0].hist(sdes[sdes < 20], bins=100, color='steelblue', alpha=0.7)
axes[0].axvline(x=cfg.SDE_DISCARD, color='orange', ls='--', label=f'SDE={cfg.SDE_DISCARD} (discard)')
axes[0].axvline(x=cfg.SDE_SUBTHRESHOLD, color='red', ls='--', label=f'SDE={cfg.SDE_SUBTHRESHOLD} (full pipeline)')
axes[0].set_xlabel('SDE'); axes[0].set_ylabel('Count')
axes[0].set_title(f'SDE Distribution (n={len(df_all)} signals)')
axes[0].legend()

# Period histogram (SDE≥7)
fp = df_all[df_all['disposition'] == cfg.DISP_FULL_PIPELINE]
axes[1].hist(fp['period'].clip(0, 30), bins=60, color='salmon', alpha=0.7)
axes[1].set_xlabel('Period (days)'); axes[1].set_ylabel('Count')
axes[1].set_title(f'Period Distribution — SDE≥7 candidates (n={len(fp)})')

plt.tight_layout()
plt.savefig('/kaggle/working/tls_summary.png', dpi=150)
plt.show()
print('\nFinal output: Upload /kaggle/working/outputs/ as "tess-tls-results" dataset')